# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.6 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente creado de forma correcta')

Cliente creado de forma correcta


### **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "Hola, ¿cómo estás?"
print(prompt)

Hola, ¿cómo estás?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

**Hola, estoy bien, gracias.** Me alegra poder ayudarte con cualquier pregunta o inquietud que tengas. ¿En qué puedo asistirte hoy?


### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print(response.usage)
print("Tokens de entrada:", response.usage.prompt_tokens)
print("Tokens de salida:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)

print("Tiempo de generación:", response.usage.completion_time)
print("Tiempo de procesamiento del prompt:", response.usage.prompt_time)
print("Tiempo en cola:", response.usage.queue_time)
print("Tiempo total:", response.usage.total_time)

CompletionUsage(completion_tokens=37, prompt_tokens=43, total_tokens=80, completion_time=0.110496725, completion_tokens_details=None, prompt_time=0.003958132, prompt_tokens_details=None, queue_time=0.053323367, total_time=0.114454857)
Tokens de entrada: 43
Tokens de salida: 37
Tokens totales: 80
Tiempo de generación: 0.110496725
Tiempo de procesamiento del prompt: 0.003958132
Tiempo en cola: 0.053323367
Tiempo total: 0.114454857


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time

inicio = time.perf_counter()

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

fin = time.perf_counter()

tiempo_respuesta = fin - inicio

print(f"Tiempo de respuesta: {tiempo_respuesta:.4f} segundos")

Tiempo de respuesta: 0.2777 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
import time

# Modelo ligero
inicio = time.perf_counter()

response_ligero = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

tiempo_ligero = time.perf_counter() - inicio


# Modelo grande
inicio = time.perf_counter()

response_grande = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

tiempo_grande = time.perf_counter() - inicio


print("MODELO LIGERO")
print("Respuesta:", response_ligero.choices[0].message.content)
print(f"Tiempo: {tiempo_ligero:.4f} segundos")
print(f"Tokens: {response_ligero.usage.total_tokens}")

print("\n" + "="*50 + "\n")

print("MODELO GRANDE")
print("Respuesta:", response_grande.choices[0].message.content)
print(f"Tiempo: {tiempo_grande:.4f} segundos")
print(f"Tokens: {response_grande.usage.total_tokens}")

MODELO LIGERO
Respuesta: Hola. Estoy funcionando correctamente. ¿En qué puedo ayudarte hoy?
Tiempo: 0.1831 segundos
Tokens: 60


MODELO GRANDE
Respuesta: **Hola, estoy bien, gracias**

Me alegra que hayas iniciado una conversación conmigo. ¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta o necesitas ayuda con algo en particular? Estoy aquí para ayudarte en lo que necesites.

Por cierto, si prefieres comunicarte en español, no hay problema. Estoy diseñado para entender y responder en varios idiomas, incluyendo el español. ¿Qué te parece si charlemos un rato?
Tiempo: 0.5008 segundos
Tokens: 148


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [ ]:
# Definir la lista de preguntas
prompts = [
    '¿Cuales son los indicios en un dataset para aplicar un análisis y modelado binomial?',
    '¿Cuál función de enlace prefieres entre logit, probit o clog-log?',
    '¿Cual es el procedimiento estandar para predictive maintenance con sensores a partir de series de tiempo?'
]


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
response_ligero = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompts[0]}
    ]
)
resultado_1 = response_ligero.choices[0].message.content

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
response_ligero = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompts[1]}
    ]
)
resultado_2 = response_ligero.choices[0].message.content

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
response_ligero = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompts[2]}
    ]
)
resultado_3 = response_ligero.choices[0].message.content

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = [resultado_1, resultado_2, resultado_3]

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
for i, res in enumerate(resultados):
    print(f'Prompt {i}')
    print(res)

Prompt 0
Para aplicar un análisis y modelado binomial en un dataset, hay varios indicios que puedes buscar. A continuación, te presento algunos de los más comunes:

1. **Variable respuesta binaria**: Un análisis binomial requiere una variable respuesta que tenga solo dos categorías, como sí/no, presente/ausente, éxito/fallo, etc.
2. **Muestra aleatoria**: La muestra debe ser aleatoria y representativa de la población objetivo.
3. **Proporción de eventos**: Existe una proporción de eventos binarios de interés en la muestra (por ejemplo, la proporción de personas que responden sí a una pregunta).
4. **Independencia de los eventos**: Los eventos binarios deben ser independientes entre sí (es decir, la probabilidad de un evento no depende de otros eventos).
5. **Homogeneidad de los eventos**: La proporción de eventos binarios debe ser constante en la muestra (es decir, no hay tendencias ni patrones en la proporción de eventos).
6. **Grandeza de la muestra**: La muestra debe ser lo suficien

Para efectos prácticos, podemos decir que respondió el modelo bien ante las peticiones. Está Ok.